In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorboard as tb 
import pytorch_lightning as pl


from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from IPython.core.display import HTML

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

from my_funs import data_processing , get_training , countious_prediction

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

In [4]:
lr_list = np.arange(1,1000 ,50) /10000 - 0.0001
lr_list = lr_list[1:]
lr_df = pd.DataFrame(columns = ['lr' , 'model_ckpt'])
lr_df['lr'] = lr_list
lr_df

,lr,model_ckpt
0,0.005,NaN
1,0.010,NaN
2,0.015,NaN
3,0.020,NaN
4,0.025,NaN
5,0.030,NaN
6,0.035,NaN
7,0.040,NaN
8,0.045,NaN
9,0.050,NaN


In [ ]:
epoch = 100
hidden_size = 32
atten_head = 4
batch_size= 32
pl.seed_everything(42)

for idx in lr_df.index:
    print(idx)
    ewma = '../2022_11_02_TEST_ewma/alpha1/04.csv'
    lr = lr_df['lr'].iloc[idx]
    print(idx, lr)
    data = data_processing(ewma)
    train_data = data[data['time_idx']  < 8760]
    training = get_training(train_data,24, 24*7)

    dataloader = training.to_dataloader(batch_size)
    validation = TimeSeriesDataSet.from_dataset(training, train_data, predict=True, stop_randomization=True)
    train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

    early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
    lr_logger = LearningRateMonitor()  
    logger = TensorBoardLogger('lr_test')
    trainer = pl.Trainer(
        max_epochs=epoch,
        gpus=0,
        gradient_clip_val=0.1,
        limit_train_batches=20,  
        callbacks=[lr_logger],
        logger=logger,
    )


    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=lr,
        hidden_size=hidden_size,  # most important hyperparameter apart from learning rate
        # number of attention heads. Set to up to 4 for large datasets
        attention_head_size=atten_head,
        dropout=0.1,  # between 0.1 and 0.3 are good values
        hidden_continuous_size=hidden_size,  # set to <= hidden_size
        output_size=7,  # 7 quantiles by default
        loss=QuantileLoss(),
        # reduce learning rate if no improvement in validation loss after x epochs
        reduce_on_plateau_patience=4,
    )

    tft.hparams['optimizer'] = 'NAdam'

    trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
    )

    print(trainer.checkpoint_callback.best_model_path)
    lr_df.loc[idx,'model_ckpt'] = trainer.checkpoint_callback.best_model_path

Global seed set to 42


0
0 0.005


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 538   
3  | prescalers                         | ModuleDict                      | 448   
4  | static_variable_selection          | VariableSelectionNetwork        | 4.6 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 29.8 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 10.0 K
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K 
8  | stati

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

lr_test/lightning_logs/version_38/checkpoints/epoch=51-step=1040.ckpt
15
15 0.08


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 538   
3  | prescalers                         | ModuleDict                      | 448   
4  | static_variable_selection          | VariableSelectionNetwork        | 4.6 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 29.8 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 10.0 K
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K 
8  | stati

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

In [21]:
lr_df.to_csv('lr_result.csv',index=False)

In [22]:
lr_df

,lr,model_ckpt
0,0.05,lr_test/lightning_logs/version_5/checkpoints/e...
1,0.10,lr_test/lightning_logs/version_6/checkpoints/e...
2,0.15,lr_test/lightning_logs/version_7/checkpoints/e...
3,0.20,lr_test/lightning_logs/version_8/checkpoints/e...
4,0.25,lr_test/lightning_logs/version_9/checkpoints/e...
5,0.30,lr_test/lightning_logs/version_10/checkpoints/...
6,0.35,lr_test/lightning_logs/version_11/checkpoints/...
7,0.40,lr_test/lightning_logs/version_12/checkpoints/...
8,0.45,lr_test/lightning_logs/version_13/checkpoints/...
9,0.50,lr_test/lightning_logs/version_14/checkpoints/...
